In [4]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit.quantum_info import Statevector, partial_trace, entropy

# -----------------------------
# Step 1: Define local operators
# -----------------------------

# Photon operators (3-level truncation: 0,1,2 photons)
a = np.array([[0,1,0],
              [0,0,np.sqrt(2)],
              [0,0,0]])   # annihilation
adag = a.T                # creation
n_photon = adag @ a        # number operator

# Atomic operators (2-level system)
sigma_z = np.array([[1,0],[0,-1]])
sigma_plus = np.array([[0,1],[0,0]])
sigma_minus = np.array([[0,0],[1,0]])

# -----------------------------
# Step 2: Build site operators
# -----------------------------

def local_site_ops():
    """Return photon and atom operators embedded in a single site Hilbert space (dim=6)."""
    a_site = np.kron(a, np.eye(2))
    adag_site = np.kron(adag, np.eye(2))
    n_site = np.kron(n_photon, np.eye(2))
    sigma_z_site = np.kron(np.eye(3), sigma_z)
    sigma_plus_site = np.kron(np.eye(3), sigma_plus)
    sigma_minus_site = np.kron(np.eye(3), sigma_minus)
    return a_site, adag_site, n_site, sigma_z_site, sigma_plus_site, sigma_minus_site

# -----------------------------
# Step 3: Embed into 4 sites
# -----------------------------

def embed(op, site, total_sites=4, local_dim=6):
    """Embed a local operator into the full Hilbert space of total_sites."""
    ops = [np.eye(local_dim) for _ in range(total_sites)]
    ops[site] = op
    result = ops[0]
    for i in range(1,total_sites):
        result = np.kron(result, ops[i])
    return result

# Build operators for each site
a_sites, adag_sites, sigma_z_sites, sigma_plus_sites, sigma_minus_sites = [], [], [], [], []
for site in range(4):
    a_site, adag_site, n_site, sz_site, sp_site, sm_site = local_site_ops()
    a_sites.append(embed(a_site, site))
    adag_sites.append(embed(adag_site, site))
    sigma_z_sites.append(embed(sz_site, site))
    sigma_plus_sites.append(embed(sp_site, site))
    sigma_minus_sites.append(embed(sm_site, site))

# -----------------------------
# Step 4: Build Hamiltonian
# -----------------------------

def build_H(J, wc, wa, g):
    dim = 6**4
    H = np.zeros((dim, dim))
    # JC Hamiltonian for each site
    for i in range(4):
        H += wc*(adag_sites[i] @ a_sites[i]) \
             + (wa/2)*sigma_z_sites[i] \
             + g*(adag_sites[i] @ sigma_minus_sites[i] + a_sites[i] @ sigma_plus_sites[i])
    # Hopping terms (chain: 1-2, 2-3, 3-4)
    H += -J*(adag_sites[0] @ a_sites[1] + a_sites[0] @ adag_sites[1])
    H += -J*(adag_sites[1] @ a_sites[2] + a_sites[1] @ adag_sites[2])
    H += -J*(adag_sites[2] @ a_sites[3] + a_sites[2] @ adag_sites[3])
    return H

# -----------------------------
# Step 5: Entanglement entropy
# -----------------------------

def entanglement_entropy(psi):
    # Partition: sites (0,1) vs (2,3)
    rho_half = partial_trace(psi, [2,3])  # trace out sites 2 and 3
    return entropy(rho_half, base=2)

# -----------------------------
# Step 6: Sweep parameters
# -----------------------------

J_values = np.linspace(0,0.3,10)
Delta_values = np.linspace(-0.5,0.5,10)
wc = 1.0
g = 0.1

entropy_map = np.zeros((len(J_values), len(Delta_values)))

for i,J in enumerate(J_values):
    for j,Delta in enumerate(Delta_values):
        wa = wc - Delta
        H = build_H(J, wc, wa, g)
        eigvals, eigvecs = np.linalg.eigh(H)
        ground_state = eigvecs[:, 0]   # lowest energy eigenvector
        ground_state = ground_state / np.linalg.norm(ground_state)
        # four sites, each of dimension 6
        psi = Statevector(ground_state, dims=[6, 6, 6, 6])

        entropy_map[i,j] = entanglement_entropy(psi)

# -----------------------------
# Step 7: Plot phase diagram
# -----------------------------

plt.imshow(entropy_map.T, origin='lower',
           extent=[J_values[0], J_values[-1], Delta_values[0], Delta_values[-1]],
           aspect='auto', cmap='viridis')
plt.colorbar(label="Entanglement Entropy")
plt.xlabel("Hopping J")
plt.ylabel("Detuning Δ")
plt.title("4-Site JCH Phase Diagram")
plt.show()


QiskitError: 'Max qargs_l is larger than number of left qargs'